## this is me trying to rebuild my pipeline code in the [antares filter format](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/learn/structure-of-a-filter/)

this filter needs to not only find the hypervariable transients, but also create the ToO taget list. im not sure if the target list generation needs to be within the filter class itself, or if it can be done afterwards once the oids already have the tags on them. 

In [229]:
from antares_devkit.models import BaseFilter, DevKitLocus
from antares_client import search
from antares_devkit.utils import filter_report
from astropy.table import QTable, Table
import numpy as np

In [228]:
class hypervariable(BaseFilter):
    #make real channel later
    # SLACK_CHANNEL = '#hypervariable-transients'
    # TRIGGERING_SURVEY = 'lsst'

    #add required locus properties
    REQUIRED_LOCUS_PROPERTIES = []

    #add required alert properties
    REQUIRED_ALERT_PROPERTIES = []

    #add more tags if needed
    REQUIRED_TAGS = ['lsst_scimma_quality_transient'] 

    #if there is a properties that i would set w this, put it here but i dont think i need this
    OUTPUT_LOCUS_PROPERTIES = [
        {
            'name': '',
            'type': '',
            'description': ''
        }] 

    OUTPUT_LOCUS_TAGS = [
        {
            'name': 'hypervariable_transients',
            'description': 'transient expressing high variability'
        }]

    #anything put here is ran once per night when antares reboots
    # def setup(self):
        
        
    #the actual filter math goes here
    def _run(self, locus):
        import numpy as np
        import pandas as pd
        from astropy.table import Table

        bands = ['lsst_g', 'lsst_r', 'lsst_i', 'lsst_z', 'lsst_u', 'lsst_y', 
               'ztf_g', 'ztf_R', 'ztf_i']

        if locus.properties['newest_alert_magnitude'] > 21: #mag cut
            return
        # if locus.alerts[-1].properties['ant_survey'] == 4:
        #     if locus.alerts[-1].properties['lsst_diaSource_snr'] < 20: #snr cut for lsst
        #         return
        # elif locus.alerts[-1].properties['ant_survey'] == 1:
        #     if locus.alerts[-1].properties['ztf_ssnrms'] < 20: #snr cut for ztf
        #         return

        #making table to organize data
        oid = Table(names = ('mjd', 'band', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg'),
                   dtype = ('f8', 'U2', 'f8', 'f8', 'f8', 'i4'))

        #retreiving and organizing alert data
        for alert in locus.alerts:
            if alert.properties['ant_survey'] == 4: #lsst alert
                #quality check
                quality = (alert.properties['lsst_diaSource_reliability'] >= 0.6)
                quality &= alert.properties['lsst_diaSource_psfFlux_flag'] == False
                quality &= alert.properties['lsst_diaSource_centroid_flag'] == False
                quality &= alert.properties['lsst_diaSource_shape_flag'] == False
                quality &= alert.properties['lsst_diaSource_isDipole'] == False
                quality &= alert.properties['lsst_diaSource_pixelFlags_saturated'] == False
                quality &= alert.properties['lsst_diaSource_pixelFlags_edge'] == False
                quality &= alert.properties['lsst_diaSource_pixelFlags_cr'] == False
                quality &= alert.properties['lsst_diaSource_pixelFlags_streak'] == False
                if 'lsst_diaSource_ssObjectId' in alert.properties:
                    quality &= alert.properties['lsst_diaSource_ssObjectId'] in (0, None, "0")
                if not quality:
                    continue

                #passed quality check, retrieve alert data
                bandpass = alert.properties['lsst_diaSource_band']
                band = 'lsst_' + str(bandpass)
                obs_date = float(alert.properties['lsst_diaSource_midpointMjdTai'])
                oid_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_scienceFlux']) + 31.4)
                temp_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_templateFlux']) + 31.4)
                diff_mag = float(temp_mag - oid_mag)
                if alert.properties['lsst_diaSource_isNegative'] == True:
                    pos_or_neg = -1 #change to be diff_mag = -diff_mag and delete else?
                else:
                    pos_or_neg = 1
                # print(band, oid[band], oid['ztf_R'])
                oid.add_row((obs_date, band, oid_mag, temp_mag, diff_mag, pos_or_neg))
                # print(band, oid[band], oid['ztf_R'])
                
            if alert.properties['ant_survey'] == 1: #ztf non limiting mag alert
                #quality check
                quality = (alert.properties['ztf_rb'] >= 0.65)
                quality &= alert.properties['ztf_nbad'] == 0
                quality &= alert.properties['ztf_fwhm'] <= 5
                quality &= alert.properties['ztf_elong'] <= 1.2
                quality &= abs(alert.properties['ztf_magdiff']) <= 0.1
                if not quality:
                    continue

                #passed quality check, retrieve alert data
                bandpass = alert.properties['ant_passband']
                band = 'ztf_' + str(bandpass)
                obs_date = float(alert.properties['ant_mjd'])
                ant_mag = alert.properties['ant_mag']
                temp_mag = alert.properties['ztf_magnr']
                diff_img_mag = alert.properties['ztf_magpsf']
                if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
                    diff_mag = -2.5* np.log10(1 - 10.**(0.4*(temp_mag - diff_img_mag)))
                    pos_or_neg = -1 #change to be diff_mag = -diff_mag ?
                else:
                    diff_mag = -2.5* np.log10(10.**(0.4*(temp_mag - diff_img_mag)) + 1)
                    pos_or_neg = 1
                oid_mag = diff_mag + temp_mag
                oid.add_row((obs_date, band, oid_mag, temp_mag, diff_mag, pos_or_neg))
        
        #finished organizing all quality alert data, calculate parameters
        for band in bands:
            band_i = oid['band'] != band

            if len(band_i) - sum(band_i) < 3: #require 3+ alerts in a given band
                continue

            oid_band = Table(oid, masked = True)
            oid_band['mjd'].mask = band_i
            oid_band['band'].mask = band_i
            oid_band['oid_mag'].mask = band_i
            oid_band['diff_mag'].mask = band_i
            oid_band['pos/neg'].mask = band_i
            
            # print(band, len(band_i)-sum(band_i), oid_band)
            
            var = np.var(oid_band['oid_mag'])
            med_diff = np.median(oid_band['diff_mag'])
            hyper_v = np.sqrt(med_diff**2 + (4*var)**2)
            print(hyper_v, var, med_diff, band)
            if hyper_v > 1:
                locus.tag('test')

In [226]:
filter_instance = hypervariable()
locus_test = search.get_by_id('ANT2020ako4i')
locus = DevKitLocus.model_validate(locus_test.to_devkit())
# filter_instance.run(locus)
report = filter_report(filter_instance, locus)
print(report)

1.3058937826059924 0.3211047335665249 0.2358528624963533 ztf_g
-- 0.14829940839171304 -- ztf_R
-- 0.07713921392828553 -- ztf_i
{'locus_id': 'ANT2020ako4i', 'time': 0.05472349998308346, 'new_locus_properties': [], 'new_tags': ['test']}


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:867: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedColumn.
  a.partition(kth, axis=axis, kind=kind, order=order)


In [58]:
filter_instance = hypervariable()
client_locus = search.get_random_locus()
locus = DevKitLocus.model_validate(client_locus.to_devkit())
filter_instance.run(locus)

a
[DevkitAlert(id='ztf_upper_limit:ZTF18acskaqh-671367714915', mjd=58425.36771989986, properties={'ztf_jd': 2458425.8677199, 'ztf_fid': 1, 'ztf_pid': 671367714915, 'ztf_diffmaglim': 20.803199768066406, 'ztf_pdiffimfilename': '/ztf/archive/sci/2018/1103/367685/ztf_20181103367685_000701_zg_c13_o_q2_scimrefdiffimg.fits.fz', 'ztf_programpi': 'Kulkarni', 'ztf_programid': 1, 'ztf_nid': 671, 'ztf_rcid': 49, 'ztf_field': 701, 'ztf_rbversion': 't12_f5_c3', 'ant_mjd': 58425.36771989986, 'ant_time_received': 1604238328, 'ant_input_msg_time': 1604212919, 'ant_passband': 'g', 'ant_maglim': 20.803199768066406, 'ant_survey': 2}, non_persistent_properties={}, grav_wave_events=[]), DevkitAlert(id='ztf_upper_limit:ZTF18acskaqh-672352384915', mjd=58426.35238429997, properties={'ztf_jd': 2458426.8523843, 'ztf_fid': 1, 'ztf_pid': 672352384915, 'ztf_diffmaglim': 20.709999084472656, 'ztf_pdiffimfilename': '/ztf/archive/sci/2018/1104/352373/ztf_20181104352373_000701_zg_c13_o_q2_scimrefdiffimg.fits.fz', 'ztf_p

FilterReturn(status='Succeeded', exception_name=None, exception_message=None, traceback=None)

In [43]:
filter_instance = hypervariable()
client_loci_ids = search.get_random_locus_ids(10)
for locus_id in client_loci_ids:
    client_locus = search.get_by_id(locus_id)
    locus = DevKitLocus.model_validate(client_locus.to_devkit())
    filter_instance.run(locus)

a


In [205]:
a = Table()
a['1'] = ['b', 'b', 'c']
a['2'] = [1, 2, 3]
a['3'] = [100, 0, 200]
a = Table(a, masked = True)
# a.add_row(('a', 4, 1000))
x = a['1'] != 'b'
a['1'].mask = x
a['2'].mask = x
a['3'].mask = x
# np.mean(a['2'])
print(len(x), sum(x), x)

3 1 [False False  True]


In [227]:
mag_max = 21
# mag_min = 17

q_lsst_magcut = {
    "query": {
        "bool": {
            "filter": [
                { "range": { "properties.newest_alert_magnitude": { "lte": mag_max} } },
                { "exists": { "field": "properties.survey.lsst" } },
                { "range": { "properties.newest_alert_observation_time": { "gte": 61234} } }
                  ]
        }
    }
}

loci = []
oids_sample = {}
# oids_var[0.4] = {}
checked = 0
for locus in search.search(q_lsst_magcut):
    oid_dv = locus.to_devkit()
    oid_locus = DevKitLocus.model_validate(oid_dv)

    oid_here = oid_get_info(oid_locus.alerts)
    oid_here['locus'] = locus
    oids_sample[oid_dv['id']] = oid_here
    # loci.append(oid_locus)
    
    # high_var_oid = oid_var_check(oid_locus.alerts, 20, 0.4)

    # if high_var_oid == None:
    #     continue

    # oids_var[0.4][oid_dv['id']]= high_var_oid
    checked += 1
    if len(oids_sample) >= 8000:
        break

NameError: name 'oid_get_info' is not defined